In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
from scipy.signal import medfilt


In [ ]:
#for folder json, open the folder of each session, then open each json file whose name does not contain aggregate

#set the path to the folder containing the json files
json_path = '../2024_study3_data/json/'


#make empty dataframe
df = pd.DataFrame(columns=['participant_number','p_position','internal_p_number','rated_p','internal_rated_number','session','timecode', 'frame_number', 'annotation_value','trigger'])
p_dict = {}


session_list = os.listdir(json_path)
for session in session_list:
    #print(session)
    path_s = json_path + session
    files = os.listdir(path_s)
    files = [f for f in files if 'aggregate' not in f]
    #one by one, open json files in a folder, read them and add them to pandas dataframe as two columns: 'time' and 'value'
    for file in files:
        
        p_number = file[12:-17]
        
        print(p_number)

        # Opening JSON file
        f = open(path_s + '/' + file)
        data = json.load(f)
        #adding to pandas df, one column for time==keys and one for values, but a single row
        values = list(data.values())
        p_position = values[0]
        print(p_position)
        p_internal_number = values[1]
        p_dict[p_internal_number] = p_position
        dict_values_1 = values[7][0]
        dict_values_2 = values[7][1]
        #print(dict_values_2)
        #get annotations
        #print(dict_values['annotations'])

        #get the annotations values
        annot_dict_1 = dict_values_1['annotations']
        rated_1 = list(dict_values_1['owner_ids'])[0]
        rated_2 = list(dict_values_2['owner_ids'])[0]
        annot_dict_2 = dict_values_2['annotations']

        
        rated_p_1 = rated_1
        internal_rated_number_1 = rated_1
        rated_p_2 = rated_2
        internal_rated_number_2 = rated_2


        #annotations dataist of dictionaries, each dictionary has the following structure "timecode": 181.133112, "frame_number": 2717, "annotation_value": 0.0, "trigger": "input"
        #get the values of the annotations
        for annot in annot_dict_1:
            df_row = pd.DataFrame(columns=['participant_number','p_position','internal_p_number','rated_p','internal_rated_number','session','timecode', 'frame_number', 'annotation_value','trigger'], index=[0])
            df_row.loc[0] = [p_number, p_position , p_internal_number, rated_p_1, internal_rated_number_1, session, annot['timecode'], annot['frame_number'], annot['annotation_value'], annot['trigger']]
            df = pd.concat([df, df_row], axis=0)
        for annot in annot_dict_2: 
            df_row = pd.DataFrame(columns=['participant_number','p_position','internal_p_number','rated_p','internal_rated_number','session','timecode', 'frame_number', 'annotation_value','trigger'], index=[0])
            df_row.loc[0] = [p_number, p_position , p_internal_number, rated_p_2, internal_rated_number_2, session, annot['timecode'], annot['frame_number'], annot['annotation_value'], annot['trigger']]
            df = pd.concat([df, df_row], axis=0)

            #closing
        f.close()

print(p_dict)

#now, change the rated_p according to the dictionary
df['rated_p'] = df['rated_p'].replace(p_dict)
#make new column p_position_rated_p
df['rater_rated'] = df['p_position']+'_' + df['rated_p']
#reset index
df.index = range(len(df))
#save df as csv
df.to_csv('study3_json.csv', index=False)


df

In [ ]:
#now, for each session, plot the annotation of each of the participants
#first, get the list of sessions
sessions = df['session'].unique()
print(sessions)

#make font bigger
plt.rcParams.update({'font.size': 18})

#3 colots
colors = ['red', 'blue', 'green']


#then, for each session, get the list of participants
for session in sessions:
    participants = df[df['session'] == session]['rater_rated'].unique()
    
    participants_list = np.unique([p[0] for p in participants])
    print(participants)
    #make plot size 20,20
    fig, ax = plt.subplots(3,1,figsize=(15,20))
    for i,participant in enumerate(participants_list):
        #get the data for the participant

        print(participant)
        rates = [part for part in participants if part[0] == participant[0]]
        
        data = df[(df['session'] == session) & (df['rater_rated']==rates[0])] 
        print(rates)
        #plot the data, as step functions
        
        ax[i].step(data['timecode'], data['annotation_value'], label=rates[0],linewidth=2, color=colors[i], alpha=0.7)
        data = df[(df['session'] == session) & (df['rater_rated']==rates[1])]
        ax[i].step(data['timecode'], data['annotation_value'], label=rates[1],linewidth=2, linestyle='--', color=colors[i], alpha=0.7)

    
        #one horizontal dotted line at y=0
        ax[i].axhline(y=0, color='black', linestyle='--')
        ax[i].legend()
        ax[i].set_xlabel('Time (s)')
        ax[i].set_ylabel('Interpersonal Ratings')
        ax[i].set_xlim(0)
        #ax[i].tight_layout()
    plt.suptitle('Interpersonal Ratings for Session ' + session)
    #make tight, start at 0
    #yaxis is "Interpersonal Ratings"
    plt.tight_layout()
    plt.show()
    

In [ ]:
df

In [ ]:
# now for each session, interpolate the values to have the same frame_seconds for all participants
import scipy
from scipy.interpolate import interp1d
#create new df with columns: session, participant, frame_seconds, value
df_interp = pd.DataFrame(columns=['participant_number','p_position','internal_p_number','rated_p','internal_rated_number','session','frame_seconds', 'value'])

#first, get session df
sessions = df['participant_number'].unique()
for session in sessions:
    df_session = df[df['participant_number']==session]
    print('length df session',len(df_session))
    print(df_session.iloc[-1])
    
    frame_seconds = np.array(df_session['timecode'].values)

    values = np.array(df_session['annotation_value'].values)
    #now, interpolate both arrays to have the frames_seconds as the same at a 0.1 second interval
    #get min and max frame_seconds
    
    frame_seconds = np.array(frame_seconds, dtype=float)
    values = np.array(values, dtype=float)

    min_frame_seconds = min(frame_seconds)
    max_frame_seconds = np.round(max(frame_seconds),1) 

    #create new frame_seconds array
    new_frame_seconds = np.arange(0.0,max_frame_seconds,0.5) 
    print(len(values))
    print(len(frame_seconds))


    print(session,min_frame_seconds,max_frame_seconds,len(new_frame_seconds),len(frame_seconds))
    

    #interpolate values (step interpolation)
    #Create the interpolation function
    f = interp1d(frame_seconds, values, kind='next',fill_value='extrapolate')

    #f = interp1d(new_frame_seconds, values, kind='next',fill_value='extrapolate')
    # Use the interpolation function to get the corresponding y-values
    new_values = f(new_frame_seconds)

    #now, add column where it is the derivative (whether value went up or down or remained the same)
    #get the derivative of the values
    #first, get the difference between the values
    diff_values = np.diff(new_values)
    #now, get the sign of the difference
    sign_diff_values = np.sign(diff_values)
    #add 0 at the beginning
    sign_diff_values = np.insert(sign_diff_values,0,0)
    diff_values = np.insert(diff_values,0,0)

    #print('new_session')
    #print(len(new_frame_seconds),len(new_values))
    #print(len(frame_seconds),len(values))
    
    #add to new df
    df_row = pd.DataFrame(columns=['participant_number','p_position','internal_p_number','rated_p','internal_rated_number','session','frame_seconds', 'value', 'diff_value','sign_diff_value'])
    df_row['frame_seconds'] = new_frame_seconds
    df_row['value'] = new_values
    df_row['diff_value'] = diff_values  
    df_row['sign_diff_value'] = sign_diff_values
    df_row['session'] = session
    df_row['participant_number'] = df_session['participant_number'].values[0]
    df_row['p_position'] = df_session['p_position'].values[0]
    df_row['internal_p_number'] = df_session['internal_p_number'].values[0]
    df_row['rated_p'] = df_session['rated_p'].values[0]
    df_row['internal_rated_number'] = df_session['internal_rated_number'].values[0]
    df_interp = pd.concat([df_interp, df_row], axis=0)

df_interp.index = range(len(df_interp))
df_interp.to_csv('study3_json_interp.csv', index=False)

df_interp



In [ ]:
#now, take the values and see how many positive and negative values we have. print that number and the ratio. Plot the distribution of the values

all_values = df_interp['value'].values
positive_values = all_values[all_values>=0]
negative_values = all_values[all_values<0]
print('Number of positive values',len(positive_values))
print('Number of negative values',len(negative_values))
print('Ratio of positive to negative values',len(positive_values)/len(negative_values))
plt.hist(all_values, bins = 15)
plt.xlabel('Interpersonal Ratings')
plt.ylabel('Number of Ratings')
plt.title('Distribution of Interpersonal Ratings')
plt.show()

#now, do the same but for sign_diff_values  
all_values = df_interp['sign_diff_value'].values
positive_values = all_values[all_values>0]
negative_values = all_values[all_values<0]
zero_values = all_values[all_values==0]
print('Number of positive values',len(positive_values))
print('Number of negative values',len(negative_values))
print('Number of zero values',len(zero_values))
print('Ratio of positive to negative values',len(positive_values)/len(negative_values))
plt.hist(all_values, bins = 3)
plt.xlabel('Interpersonal Ratings Sign')
plt.ylabel('Number of Ratings')
plt.title('Distribution of Interpersonal Ratings')
plt.show()

#now, exclude the 0 values and plot
all_values_minus_zero = all_values[all_values!=0]
plt.hist(all_values_minus_zero, bins = 2)
plt.xlabel('Interpersonal Ratings Sign with no Zero')
plt.ylabel('Number of Ratings')
plt.title('Distribution of Interpersonal Ratings')
plt.show()

